# HotelDataAgentRuntimePublish

Validate the draft HotelOperationsAgent definition and optionally publish the staging configuration
through the Fabric DataAgent runtime API.

In [ ]:
workspace_id = "00000000-0000-0000-0000-000000000000"
data_agent_id = "00000000-0000-0000-0000-000000000000"
capacity_id = "00000000-0000-0000-0000-000000000000"
publish_mode = "validate_only"
published_description = "HotelOperationsAgent draft publication"
native_readiness_json = "{}"

In [ ]:
from __future__ import annotations

import asyncio
import base64
import hashlib
import json
import re
import time
from uuid import UUID

import sempy.fabric as fabric

client = fabric.FabricRestClient()
base = f"/v1/workspaces/{workspace_id}/dataAgents/{data_agent_id}"
if publish_mode not in {"validate_only", "deploy"}:
    raise ValueError("publish_mode must be validate_only or deploy")

def await_operation(response, operation_name):
    if response.status_code in {200, 201}:
        return response
    if response.status_code != 202:
        raise RuntimeError(f"{operation_name} failed: {response.status_code} {response.text[:500]}")
    location = response.headers.get("Location") or response.headers.get("location")
    if not location:
        raise RuntimeError(f"{operation_name} returned 202 without a Location header")
    retry_after = int(response.headers.get("Retry-After", "5"))
    for _ in range(60):
        time.sleep(max(1, retry_after))
        poll = client.get(location)
        if poll.status_code != 200:
            raise RuntimeError(f"{operation_name} poll failed: {poll.status_code} {poll.text[:500]}")
        payload = poll.json()
        status = str(payload.get("status", "")).lower()
        if status in {"succeeded", "completed"}:
            result_response = client.get(location.rstrip("/") + "/result")
            return result_response if result_response.status_code == 200 else poll
        if status in {"failed", "cancelled"}:
            raise RuntimeError(f"{operation_name} failed: {json.dumps(payload, default=str)}")
        retry_after = int(poll.headers.get("Retry-After", retry_after))
    raise TimeoutError(f"{operation_name} did not complete within the polling limit")

def ensure_data_agent_sdk():
    try:
        from fabric.dataagent.client import FabricDataAgentManagement
        from mcp import ClientSession
        return FabricDataAgentManagement
    except ImportError as error:
        raise RuntimeError("Preinstall fabric-data-agent-sdk>=0.1.33a0,<0.2 and mcp>=1,<2 from the approved package feed. This notebook does not download dependencies.") from error

def all_source_elements(datasource, parent_id=None, stage="staging"):
    elements = []
    token = None
    seen = set()
    while True:
        page = datasource.get_elements(stage, parent_id, continuation_token=token)
        elements.extend(page["value"])
        token = page.get("continuationToken")
        if not token:
            if page.get("continuationUri"):
                raise RuntimeError("Source elements returned a continuation URI without its token")
            return elements
        if token in seen:
            raise RuntimeError("Source element pagination repeated a token")
        seen.add(token)

def named_source_element(datasource, parent_id, name):
    matches = [element for element in all_source_elements(datasource, parent_id) if element["displayName"] == name]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one schema element named {name}, found {len(matches)}")
    return matches[0]

def lakehouse_source_tables(datasource):
    schemas = named_source_element(datasource, None, "Schemas")
    dbo = named_source_element(datasource, schemas["id"], "dbo")
    tables = named_source_element(datasource, dbo["id"], "Tables")
    return all_source_elements(datasource, tables["id"])

def lakehouse_example_query(query):
    query = re.sub(r"\bDATE\s*'(\d{4}-\d{2}-\d{2})'", r"CAST('\1' AS date)", query, flags=re.IGNORECASE)
    return re.sub(r"\bCURRENT_DATE\b", "CAST(GETDATE() AS date)", query, flags=re.IGNORECASE)

def synchronize_source_examples(datasource, examples, lakehouse=False):
    current = datasource.get_fewshots("staging").to_dict("records")
    by_question = {row["question"].casefold(): row for row in current}
    if len(by_question) != len(current):
        raise RuntimeError("Data source contains duplicate example questions")
    expected = {}
    for question, query in examples.items():
        query = lakehouse_example_query(query) if lakehouse else query
        expected[question.casefold()] = query
        existing = by_question.get(question.casefold())
        if existing is None:
            datasource.add_fewshot(question, query)
        elif existing.get("query") != query:
            datasource.update_fewshot(existing["id"], question=question, query=query)
    confirmed = datasource.get_fewshots("staging").to_dict("records")
    actual = {row["question"].casefold(): row.get("query") for row in confirmed}
    if len(actual) != len(confirmed) or any(actual.get(question) != query for question, query in expected.items()):
        raise RuntimeError("Example queries did not read back exactly")
    return len(confirmed)

def model_example_guidance(examples):
    return "## DAX example guidance\n\n" + "\n\n".join(f"### {question}\n\n```dax\n{query}\n```" for question, query in examples.items())

def draft_definition_sha256(definition):
    parts = definition.get("definition", {}).get("parts", [])
    if not parts:
        raise RuntimeError("The Data Agent definition contains no parts")
    canonical = sorted(parts, key=lambda part: part["path"])
    return hashlib.sha256(json.dumps(canonical, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def require_native_readiness(raw, definition, expected_workspace, expected_agent, source_ids):
    evidence = json.loads(raw)
    if not isinstance(evidence, dict) or evidence.get("workspaceId") != expected_workspace or evidence.get("dataAgentId") != expected_agent:
        raise RuntimeError("Native UI readiness evidence is missing or targets another Data Agent")
    if evidence.get("draftDefinitionSha256") != draft_definition_sha256(definition):
        raise RuntimeError("Native UI readiness evidence does not match the current draft definition")
    def verified(component, artifact_id):
        if not isinstance(component, dict) or component.get("artifactId") != artifact_id:
            raise RuntimeError("Native UI readiness evidence has a missing or incorrect artifact binding")
        if component.get("verificationMethod") != "native-ui-readback" or not isinstance(component.get("evidenceRef"), str) or not component["evidenceRef"].strip():
            raise RuntimeError("Native readiness requires independently reviewed UI evidence, not stored source instructions")
        digest = component.get("evidenceSha256", "")
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest) or digest == "0" * 64:
            raise RuntimeError("Native UI readiness evidence requires its retained artifact SHA256")
    verified(evidence.get("ontologyContext"), source_ids["HotelOperationsOntology"])
    topics = evidence.get("sqlTopics", {})
    if not isinstance(topics, dict) or set(topics) != {"HotelOperations", "HotelGuestServices"}:
        raise RuntimeError("Both SQL sources require native Topics readback evidence")
    for name in topics:
        verified(topics[name], source_ids[name])
    model = evidence.get("modelPrepForAI")
    verified(model, source_ids["HotelOperationsModel"])
    count = model.get("verifiedAnswerCount")
    if model.get("instructionsConfigured") is not True or type(count) is not int or count < 10:
        raise RuntimeError("Model-owned Prep for AI requires instructions and at least 10 verified answers; stored instruction guidance does not satisfy this gate")
    return evidence

def verify_existing_source(datasource, display_name, config, stage="staging"):
    metadata = datasource.get_configuration(stage)
    if metadata.get("displayName") != display_name or metadata.get("selectionState") != "Selected" or len(str(metadata.get("description", "")).split()) < 10:
        raise RuntimeError(f"{display_name} has missing or mismatched native source metadata")
    if "data dictionary" not in str(metadata.get("instructions", "")).casefold():
        raise RuntimeError(f"{display_name} requires a complete source data dictionary")
    discovered = discover_source_elements(datasource, stage=stage)
    business = [element for _, element in discovered if element.get("type") in {"Table", "Column", "Measure"}]
    tables = [element for element in business if element["type"] == "Table"]
    if not tables or any(element.get("state") != "Available" or element.get("isSelected", element.get("is_selected")) is not True for element in business):
        raise RuntimeError(f"{display_name} has unavailable or unselected business elements")
    if set(config["tables"]) - {element["displayName"] for element in tables}:
        raise RuntimeError(f"{display_name} is missing required tables")
    examples = datasource.get_fewshots(stage).to_dict("records")
    if config["artifact_type"] == "Lakehouse":
        if any(len(str(element.get("description", "")).split()) < 5 for element in business):
            raise RuntimeError(f"{display_name} has missing native table or column descriptions")
        questions = {example["question"].casefold() for example in examples}
        if len(questions) != len(examples) or len(examples) < 10:
            raise RuntimeError(f"{display_name} requires at least 10 distinct native example queries")
        for example in examples:
            query = example["query"]
            if lakehouse_example_query(query) != query or re.search(r"\bLIMIT\s+\d+|\b=\s*(?:true|false)\b", query, re.IGNORECASE):
                raise RuntimeError(f"{display_name} contains a non-native SQL example")
    elif examples:
        raise RuntimeError("Semantic-model instruction examples must not be reported as native few-shots")
    snapshot = {
        "metadata": {key: metadata.get(key) for key in ("id", "displayName", "type", "description", "instructions", "selectionState", "lakehouseReference", "itemReference")},
        "elements": sorted(({key: element.get(key) for key in ("id", "displayName", "type", "description", "isSelected", "is_selected")} for element in business), key=lambda element: element["id"]),
        "examples": sorted(({key: example.get(key) for key in ("id", "question", "query")} for example in examples), key=lambda example: example["question"]),
    }
    digest = hashlib.sha256(json.dumps(snapshot, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    return {"selected_element_count": len(business), "table_count": len(tables), "fewshot_count": len(examples), "native_fewshots_supported": config["artifact_type"] == "Lakehouse", "configuration_sha256": digest}

def workspace_item_id(display_name, item_type):
    response = client.get(f"/v1/workspaces/{workspace_id}/items")
    if response.status_code != 200:
        raise RuntimeError(f"Workspace item discovery failed: {response.status_code} {response.text[:500]}")
    matches = [
        item
        for item in response.json().get("value", [])
        if item.get("displayName") == display_name and item.get("type") == item_type
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {item_type} named {display_name}, found {len(matches)}")
    return matches[0]["id"]

def source_configs_from_definition(response):
    configs = {}
    payload = response.json()
    parts = payload.get("definition", {}).get("parts", [])
    decoded = {}
    for part in parts:
        path = part.get("path", "")
        if not path.startswith("Files/Config/draft/"):
            continue
        if not (path.endswith("/datasource.json") or path.endswith("/fewshots.json")):
            continue
        decoded[path] = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
    for path, datasource in decoded.items():
        if not path.endswith("/datasource.json"):
            continue
        folder = path.rsplit("/", 1)[0]
        fewshots = decoded.get(folder + "/fewshots.json", {}).get("fewShots", [])
        configs[datasource["displayName"]] = {
            "description": datasource.get("userDescription", ""),
            "instructions": datasource.get("dataSourceInstructions", ""),
            "fewshots": {item["question"]: item.get("query") for item in fewshots},
        }
    return configs

agent_instructions = """Source routing:
Include ISO 8601 dates (YYYY-MM-DD) in reservation and stay answers, even when also using localized date wording. Preserve the exact source dates.
Route property identity, room inventory, rate, outlet, maintenance, and work-order questions to HotelOperations, which is authoritative for operational master and event data.
Route guest identity, reservation, stay, folio, payment, dispute, request, and reconciliation questions to HotelGuestServices, which is authoritative for guest-service records and curated facts.
Route governed occupancy, rate, revenue, booking-behavior, dispute, request, and maintenance-response KPI questions to HotelOperationsModel, and use its measures instead of recomputing them.
Use HotelOperationsOntology as semantic context for entities, relationships, and lineage. Execute record joins against the authoritative lakehouse SQL sources using the mapped keys. The ontology is not a query execution endpoint. Do not claim GQL or GraphModel execution.
For cross-source questions, query every authoritative source separately and join only on documented stable business keys. Resolve every property_id to the official HotelOperations property name. For chain-wide aggregations, start with the complete HotelOperations property roster, join matching HotelGuestServices facts, and return all 20 properties, including zero-count properties. Use 2026-09-01 as the fixed temporal boundary: past records require the relevant date to be strictly less than DATE '2026-09-01', while future or upcoming reservations require check_in strictly greater than DATE '2026-09-01' and normally use status future or arriving. Never use an inclusive boundary or classify a 2026-09-01 record as past or future; current-day records are separate and excluded from both groups. Never mislabel arrivals as newly created bookings because no booking-created timestamp exists. Never claim a cross-lakehouse write is ACID; surface pending_reconciliation from action_sagas.
Acronym compendium — hotel and accommodation glossary: `ADR` = Average Daily Rate; `ALOS` = Average Length of Stay; `BAR` = Best Available Rate; `CRS` = Central Reservation System; `F&B` = Food and Beverage; `FIT` = Free Independent Traveler; `GDS` = Global Distribution System; `GOPPAR` = Gross Operating Profit Per Available Room; `HoReCa` or `HORECA` = Hotel Restaurant and Cafe; `LOS` = Length of Stay; `MICE` = Meetings Incentives Conferences and Exhibitions; `NRevPAR` = Net Revenue Per Available Room; `OCC` = Occupancy Rate; `OTA` = Online Travel Agency; `PMS` = Property Management System; `POS` = Point of Sale; `RevPAR` = Revenue Per Available Room; `RMS` = Revenue Management System; `TRevPAR` = Total Revenue Per Available Room; `CPOR` = Cost Per Occupied Room; `OOO` = Out of Order; `OOS` = Out of Service; `DND` = Do Not Disturb; `ETA` = Estimated Time of Arrival; `ETD` = Estimated Time of Departure; `VIP` = Very Important Person; `comp` = complimentary charge or stay; `folio` = guest account statement; `room block` = rooms held for a group; `pickup` = rooms consumed from a block or pace period; `wash` = expected block attrition; `stayover` = guest remaining another night; `due-in` = expected arrival; `due-out` = expected departure.
Organization-specific acronyms: `LGR` = Lyrenza Guest Record; `LSR` = Lyrenza Service Request; `LCR` = Lyrenza Care Recovery; `SIR` = Stay Incident Record; `GPF` = Guest Preference Flag."""

source_configs = {
    "HotelGuestServices": {
        "description": "## Authority\n\nAuthoritative guest-services record for hotel guests and their service journeys.\n\n## Scope\n\nReservations, stays, dining, folios, payments, disputes, requests, confirmations, audit events, and reconciliation workflows are governed here.\n\n## Boundary\n\nOfficial property and operational facts belong to HotelOperations.",
        "artifact_type": "Lakehouse",
        "sdk_type": "lakehouse",
        "instructions": "Use this source for guest-facing records. reservation_id is the reservation confirmation key. Filter reservations.reservation_id by the full supplied value for exact lookups. Use property_id to obtain property name and city from HotelOperations in a second query. Use 2026-09-01 as the fixed temporal boundary: past reservations require check_in < DATE '2026-09-01'; future or upcoming reservations require check_in > DATE '2026-09-01' and status IN ('future','arriving'); records dated 2026-09-01 are excluded from both. The source does not contain a booking-created timestamp.",
        "tables": ["guests", "reservations", "reservation_guests", "stays", "room_assignments", "folios", "folio_transactions", "payments", "adjustments", "billing_disputes", "special_requests", "restaurant_reservations", "action_sagas", "action_confirmations", "idempotency_records"],
        "fewshots": {
            "What is Maria Garcia's upcoming reservation strictly after September 1, 2026?": "SELECT r.reservation_id, r.property_id, r.check_in, r.check_out, r.status FROM reservations r JOIN guests g ON g.guest_id = r.guest_id WHERE g.first_name = 'Maria' AND g.last_name = 'Garcia' AND r.check_in > DATE '2026-09-01' AND r.status IN ('future', 'arriving') ORDER BY r.check_in;",
            "Show future reservations arriving at HTL-004 strictly after September 1, 2026.": "SELECT reservation_id, guest_id, property_id, check_in, check_out, status, channel FROM reservations WHERE property_id = 'HTL-004' AND check_in > DATE '2026-09-01' AND status IN ('future', 'arriving') ORDER BY check_in;",
            "Which open billing disputes have the highest amounts?": "SELECT TOP (20) billing_dispute_id, folio_id, status, amount, currency FROM billing_disputes WHERE status NOT IN ('resolved','rejected') ORDER BY amount DESC;",
            "Show the completed past reservation RES-FOLIO-2026 strictly before September 1, 2026.": "SELECT reservation_id, guest_id, property_id, check_in, check_out, status FROM reservations WHERE reservation_id = 'RES-FOLIO-2026' AND check_in < DATE '2026-09-01' AND status = 'completed';",
            "What disputed minibar charge is recorded for folio FOL-0000001?": "SELECT d.folio_id, d.dispute_type, d.amount, d.status FROM billing_disputes d WHERE d.folio_id = 'FOL-0000001' AND d.dispute_type = 'minibar' AND d.amount = 15.00;",
            "What is the balance for folio FOL-0000001?": "SELECT folio_id, property_id, balance FROM folios WHERE folio_id = 'FOL-0000001';",
            "List open billing disputes in descending amount order.": "SELECT billing_dispute_id, folio_id, status, amount FROM billing_disputes WHERE status IN ('open', 'under_review') ORDER BY amount DESC;",
            "Show reservation channels represented at HTL-004.": "SELECT channel, COUNT(*) AS reservation_count FROM reservations WHERE property_id = 'HTL-004' GROUP BY channel ORDER BY reservation_count DESC;",
            "Which reconciliation workflows are not complete?": "SELECT saga_id, state, pending_reconciliation FROM action_sagas WHERE pending_reconciliation = 1 OR state <> 'completed' ORDER BY saga_id;",
            "Count reservations by lifecycle status.": "SELECT status, COUNT(*) AS reservation_count FROM reservations GROUP BY status ORDER BY reservation_count DESC;",
        },
    },
    "HotelOperations": {
        "description": "## Authority\n\nAuthoritative operational record for the complete hotel property estate.\n\n## Scope\n\nProperty, room, amenity, outlet, inventory, rate, maintenance, work-order, and daily operating facts are governed here.\n\n## Boundary\n\nGuest identity and guest-service records belong to HotelGuestServices.",
        "artifact_type": "Lakehouse",
        "sdk_type": "lakehouse",
        "instructions": "Use this source for property and operational facts. properties.property_id is the shared key used by HotelGuestServices. For cross-source questions, resolve property_id from the guest record and then query properties for name, city, region, country, and currency. For chain-wide aggregations, query the complete property roster so zero-count properties remain in the final answer.",
        "tables": ["properties", "room_types", "rooms", "amenities", "property_amenities", "outlets", "daily_inventory", "daily_rates", "rate_plans", "property_daily_metrics", "maintenance_requests", "work_orders"],
        "fewshots": {
            "How many properties are in the chain?": "SELECT COUNT(*) AS property_count FROM properties;",
            "What are the name and city of property HTL-004?": "SELECT property_id, name, city, region, country, currency FROM properties WHERE property_id = 'HTL-004';",
            "Which maintenance requests are still open for HTL-004?": "SELECT maintenance_request_id, property_id, room_id, request_type, status, reported_at FROM maintenance_requests WHERE property_id = 'HTL-004' AND status <> 'resolved' ORDER BY reported_at;",
            "List the complete property roster for a chain-wide result.": "SELECT property_id, name, city, region FROM properties ORDER BY property_id;",
            "Show daily occupancy inputs for HTL-004 in July.": "SELECT service_date, occupied_rooms, sellable_rooms FROM property_daily_metrics WHERE property_id = 'HTL-004' AND service_date BETWEEN DATE '2026-07-01' AND DATE '2026-07-31' ORDER BY service_date;",
            "Which properties have the highest room revenue today?": "SELECT property_id, room_revenue FROM property_daily_metrics WHERE service_date = CURRENT_DATE ORDER BY room_revenue DESC;",
            "Count open maintenance requests by property.": "SELECT property_id, COUNT(*) AS open_request_count FROM maintenance_requests WHERE status <> 'resolved' GROUP BY property_id ORDER BY open_request_count DESC;",
            "Show urgent work orders and their originating maintenance requests.": "SELECT work_order_id, maintenance_request_id, priority, opened_at FROM work_orders WHERE priority IN ('urgent', 'high') ORDER BY opened_at;",
            "What currencies are used by properties in each region?": "SELECT region, currency, COUNT(*) AS property_count FROM properties GROUP BY region, currency ORDER BY region, currency;",
            "Show unresolved maintenance requests in report-time order.": "SELECT maintenance_request_id, property_id, status, reported_at FROM maintenance_requests WHERE status <> 'resolved' ORDER BY reported_at;",
            "What parking, valet price, transportation, and pool hours apply at Lyrenza Scottsdale Sonoran Resort?": "SELECT property_id, name, parking, transport, pool_hours FROM properties WHERE property_id = 'HTL-004';",
        },
    },
}

isolation_checks = [
    {"name": "known-folio-minibar-dispute", "question": "What disputed minibar charge is recorded for folio FOL-0000001?", "expected_terms": ["15.00", "minibar"]},
    {"name": "official-property-name-routing", "question": "What is the official property name for the hotel identified as HTL-004?", "expected_terms": ["Lyrenza Scottsdale Sonoran Resort"]},
    {"name": "scottsdale-parking-transport-hours", "question": "What parking, valet price, transportation, and pool hours apply at Lyrenza Scottsdale Sonoran Resort?", "expected_terms": ["25", "valet", "shuttle", "07:00", "21:00"]},
    {"name": "past-reservation-strict-boundary", "question": "Using the September 1, 2026 temporal boundary, identify the completed past reservation RES-FOLIO-2026 and report its dates.", "expected_terms": ["RES-FOLIO-2026", "2026-03-03", "2026-03-05"]},
    {"name": "future-maria-reservation-strict-boundary", "question": "Using the September 1, 2026 temporal boundary, identify Maria Garcia's nearest upcoming reservation. Include its reservation identifier, official property name, internal property identifier, check-in date, check-out date, and status.", "expected_terms": ["RES-FAMILY-2026", "Lyrenza Scottsdale Sonoran Resort", "HTL-004", "2026-09-10", "2026-09-12", "future"]},
    {"name": "semantic-model-completed-stays", "question": "Using HotelOperationsModel, what is the chainwide Completed Stay Count?", "expected_terms": ["117,539"]},
    {"name": "ontology-guest-reservation-relationship", "question": "Using HotelOperationsOntology, which relationship connects a Guest to a Reservation?", "expected_terms": ["GuestBookedReservation"]},
]

def element_dictionary(datasource, table_names, tables_by_name):
    lines = ["## Complete data dictionary", "", "Synchronized from every table exposed by the connected source.", ""]
    for table_name in table_names:
        table = tables_by_name[table_name]
        columns = all_source_elements(datasource, table["id"])
        field_entries = []
        for column in columns:
            field_name = column.get("displayName")
            if field_name:
                field_type = column.get("dataType") or column.get("type") or "field"
                field_entries.append(f"`{field_name}` ({field_type})")
        lines.append(f"### `{table_name}`")
        lines.append("")
        lines.append("Fields: " + ", ".join(field_entries) + ".")
        lines.append("")
    return "\n".join(lines)

def discover_source_elements(datasource, stage="staging"):
    discovered = []
    queue = [(None, tuple())]
    seen = set()
    while queue:
        parent_id, parent_path = queue.pop(0)
        children = all_source_elements(datasource, parent_id, stage=stage)
        for element in children:
            element_id = element.get("id")
            if not element_id or element_id in seen:
                continue
            seen.add(element_id)
            path = parent_path + (element.get("displayName", element_id),)
            discovered.append((path, element))
            if element.get("hasSubElements") is True:
                queue.append((element_id, path))
    return discovered

def generic_element_dictionary(discovered):
    lines = ["## Complete data dictionary", "", "Synchronized from every element exposed by the connected source.", ""]
    for path, element in discovered:
        element_type = element.get("type") or "element"
        lines.append(f"- `{' > '.join(path)}` ({element_type})")
    return "\n".join(lines)

async def query_published_agent(question):
    from datetime import timedelta
    from mcp import ClientSession
    from mcp.client.streamable_http import streamablehttp_client
    token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
    endpoint = f"https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}/dataagents/{data_agent_id}/agent"
    async with streamablehttp_client(endpoint, headers={"Authorization": f"Bearer {token}"}, timeout=300) as (read, write, _), ClientSession(read, write) as session:
        await session.initialize()
        tools = (await session.list_tools()).tools
        if len(tools) != 1:
            raise RuntimeError(f"Published Data Agent isolation expected one MCP tool, found {len(tools)}")
        tool = tools[0]
        argument = next(iter(tool.inputSchema["properties"]))
        response = await session.call_tool(tool.name, {argument: question}, read_timeout_seconds=timedelta(seconds=300))
        text = "\n".join(block.text for block in response.content if block.type == "text").strip()
        if response.isError or not text:
            raise RuntimeError(f"Published Data Agent isolation query failed: {text}")
        return text

async def run_isolation_checks(checks):
    results = []
    for check in checks:
        answer = await query_published_agent(check["question"])
        missing = [term for term in check["expected_terms"] if term.casefold() not in answer.casefold()]
        if missing:
            raise RuntimeError(f"Published-agent verification {check['name']} is missing expected terms: {missing}")
        results.append({"name": check["name"], "status": "passed"})
    return results

definition = client.post(base + "/getDefinition", json={})
definition = await_operation(definition, "DataAgent getDefinition")
definition_configs = source_configs_from_definition(definition)
if set(definition_configs) != {"HotelOperations", "HotelGuestServices", "HotelOperationsModel"}:
    raise RuntimeError("Expected the two native lakehouse sources and semantic model. Ontology V2 must be attached as context, not a legacy graph datasource.")
source_configs["HotelOperationsModel"] = {**definition_configs["HotelOperationsModel"], "artifact_type": "SemanticModel", "tables": []}
result = {"workspace_id": workspace_id, "data_agent_id": data_agent_id, "capacity_id": capacity_id, "publish_mode": publish_mode, "published_description": published_description, "validation": {"definition_status_code": definition.status_code}}
result["draft_definition_sha256"] = draft_definition_sha256(definition.json())
if publish_mode == "deploy":
    source_ids = {name: workspace_item_id(name, config["artifact_type"]) for name, config in source_configs.items()}
    source_ids["HotelOperationsOntology"] = workspace_item_id("HotelOperationsOntology", "Ontology")
    result["native_readiness_evidence"] = require_native_readiness(native_readiness_json, definition.json(), workspace_id, data_agent_id, source_ids)
    FabricDataAgentManagement = ensure_data_agent_sdk()
    management = FabricDataAgentManagement(UUID(data_agent_id), UUID(workspace_id))
    staging_settings = management.get_settings("staging")
    if staging_settings.get("experimental", {}).get("enableExperimentalFeatures") is not True:
        raise RuntimeError("The Data Agent preview runtime was not enabled")
    sources = management.list_datasources("staging")
    if len(sources) != 3:
        raise RuntimeError("The native staging source set changed before publication")
    verified_sources = {}
    for display_name, config in source_configs.items():
        matches = [source for source in sources if source.get_configuration("staging").get("displayName") == display_name]
        if len(matches) != 1 or matches[0].get_configuration("staging").get("id") != source_ids[display_name]:
            raise RuntimeError(f"{display_name} is missing, duplicated, or bound to another artifact")
        verified_sources[display_name] = verify_existing_source(matches[0], display_name, config)
    latest_definition = await_operation(client.post(base + "/getDefinition", json={}), "Recheck DataAgent definition")
    if draft_definition_sha256(latest_definition.json()) != result["draft_definition_sha256"]:
        raise RuntimeError("The draft changed during verification; no publication was submitted")
    publish = management.publish_staging(description=published_description)
    result["verified_sources"] = verified_sources
    result["publish"] = publish
    published_settings = management.get_settings("published")
    for key in ("aiInstructions", "experimental"):
        if key not in staging_settings or published_settings.get(key) != staging_settings[key]:
            raise RuntimeError(f"Published {key} does not match the verified staging settings")
    published_sources = management.list_datasources("published")
    if len(published_sources) != 3:
        raise RuntimeError("Published native source count does not match staging")
    for display_name, config in source_configs.items():
        matches = [source for source in published_sources if source.get_configuration("published").get("displayName") == display_name]
        if len(matches) != 1 or matches[0].get_configuration("published").get("id") != source_ids[display_name]:
            raise RuntimeError(f"Published {display_name} has an incorrect artifact binding")
        if verify_existing_source(matches[0], display_name, config, stage="published") != verified_sources[display_name]:
            raise RuntimeError(f"Published {display_name} schema or examples differ from staging")
    result["isolation_checks"] = asyncio.run(run_isolation_checks(isolation_checks))

In [ ]:
notebookutils.notebook.exit(json.dumps(result, default=str))